In [32]:

from micrograd.value import ScalarValue
import random

class Neuron:
    
    def __init__(self, nin):
        self.w = [ScalarValue(random.uniform(-1,1)) for _ in range(nin)]
        self.b = ScalarValue(random.uniform(-1,1))
        

    def __call__(self, x):
        # w * x + b
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"Neuron(w={self.w}, b={self.b})"
    

In [33]:
n1 = Neuron(2)
x = [2.0, 3.0]

n1(x)

ScalarValue(data=-0.02076385675577734, grad=0)

In [54]:

class Layer:
    
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs) ==1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
        

In [55]:
layer1 = Layer(2, 3)
x = [2,4]

layer1_out = layer1(x)
layer1_out

hidden_layer1 = Layer(3, 4)
hidden_layer1_out = hidden_layer1(layer1_out)
hidden_layer1_out


[ScalarValue(data=0.5345193717060603, grad=0),
 ScalarValue(data=-0.4274682075398252, grad=0),
 ScalarValue(data=0.10303546728661597, grad=0),
 ScalarValue(data=-0.9514973003888298, grad=0)]

In [56]:

class MLP:
    
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(nin=sz[i], nout=sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [ p for layer in self.layers for p in layer.parameters()]

x = [3.0,4.0,-24.0]
mlp1 = MLP(3, [4,4,1])

mlp1(x)


ScalarValue(data=0.9031398520606083, grad=0)

In [ ]:
mlp1.parameters()

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
] # dataset

ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

# try in above MLP
ypred = [mlp1(x) for x in xs]
ypred

[[ScalarValue(data=-0.8644523010249456, grad=0)],
 [ScalarValue(data=0.3143707157829178, grad=0)],
 [ScalarValue(data=-0.48543176842525415, grad=0)],
 [ScalarValue(data=-0.7033046288234961, grad=0)]]

In [53]:
loss = sum([(ypredi - ysi)**2 for ysi, ypredi in zip(ys, ypred)])
loss

TypeError: unsupported operand type(s) for -: 'list' and 'float'

In [48]:
loss.backward()